# 05 — Training: RF-DETR Bib Detector

Trains an RF-DETR Nano model on the same Roboflow-exported dataset used for YOLOv8n.

**Dataset:** `race-vision.v1i.yolov8` — 174 train / 22 val / 21 test, class: `bib`  
**Model:** RF-DETR Nano (`RFDETRNano`) — 6.5M params, DINOv2 backbone, Apache 2.0 licensed  
**Device:** MPS (Apple Silicon) auto-detected  
**Goal:** Fair comparison against the YOLOv8n baseline from `03_training.ipynb` (both ~6M params)

## Setup

In [ ]:
from pathlib import Path
import importlib.metadata
import torch

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

import rfdetr
print(f"RF-DETR version: {importlib.metadata.version('rfdetr')}")

DATASET_DIR = Path("../data/labeled/race-vision.v1i.yolov8").resolve()
OUTPUT_DIR  = Path("../models/runs/rfdetr-nano-map50").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"
print(f"Dataset: {DATASET_DIR}")
print(f"Output:  {OUTPUT_DIR}")

## Dataset sanity check

RF-DETR expects YOLO format: `dataset_dir/train/images/`, `dataset_dir/valid/images/`, with matching `labels/` dirs.
It ignores the path entries inside `data.yaml` and just checks for the directory structure.

In [ ]:
import yaml

with open(DATASET_DIR / "data.yaml") as f:
    cfg = yaml.safe_load(f)

print(f"Classes ({cfg['nc']}): {cfg['names']}")

for split in ("train", "valid", "test"):
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"  {split}: {n_img} images, {n_lbl} labels")

## Train

Key hyperparameters:
- `epochs=100` — same budget as YOLOv8n run for a fair comparison
- `batch_size=4, grad_accum_steps=4` — effective batch size of 16 (RF-DETR's target), sized for Apple M4 memory
- `imgsz=560` — RF-DETR Nano native resolution
- `early_stopping=True, early_stopping_patience=20` — matches YOLOv8n `patience=20`
- `skip_best_epochs=5` — prevents early stopping from triggering in the first 5 epochs when the pretrained model starts with artificially high val mAP before adapting to the custom dataset
- `num_workers=2` — safe default for MPS; increase on machines with more CPUs

**Checkpoint metric patch:** RF-DETR's default is to save the best checkpoint based on `val/mAP_50_95`.
For bib detection (pure detection, not tight localisation), `val/mAP_50` is the right signal.
This requires a one-line patch to the installed package since there is no public API for it:
`.venv/lib/python3.12/site-packages/rfdetr/training/trainer.py` — `BestModelCallback` and
`RFDETREarlyStopping` both have `monitor_regular="val/mAP_50"` and `monitor_ema="val/ema_mAP_50"` set.
Output goes to `rfdetr-nano-map50/` to keep separate from the original `rfdetr-nano/` run.

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano()

model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=100,
    batch_size=4,
    grad_accum_steps=4,
    imgsz=560,
    device=DEVICE,
    output_dir=str(OUTPUT_DIR),
    early_stopping=True,
    early_stopping_patience=20,
    skip_best_epochs=5,
    num_workers=2,
)

best = OUTPUT_DIR / "checkpoint_best_total.pth"
print(f"\nBest weights: {best}")
print(f"Exists: {best.exists()}")

## Training curves

RF-DETR logs to a CSV in the output directory during training. Plot loss and mAP progression.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("../models/runs/rfdetr-nano-map50").resolve()
ARTIFACTS = Path("artifacts/rfdetr-nano-map50")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "metrics.csv"
assert csv_path.exists(), f"No metrics CSV found: {csv_path}"

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# RF-DETR logs two rows per epoch: one with train metrics, one with val metrics
train_df = df[df["train/loss"].notna()].copy()
val_df   = df[df["val/mAP_50"].notna()].copy()

print(f"Epochs logged — train: {len(train_df)}, val: {len(val_df)}")
print(f"Best val mAP50:    {val_df['val/mAP_50'].max():.4f}  (epoch {int(val_df.loc[val_df['val/mAP_50'].idxmax(), 'epoch'])})")
print(f"Best val mAP50-95: {val_df['val/mAP_50_95'].max():.4f}  (epoch {int(val_df.loc[val_df['val/mAP_50_95'].idxmax(), 'epoch'])})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(train_df["epoch"], train_df["train/loss"], label="train loss")
ax.plot(val_df["epoch"],   val_df["val/loss"],     label="val loss")
ax.set_title("Loss")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(val_df["epoch"], val_df["val/mAP_50"],    label="mAP50")
ax.plot(val_df["epoch"], val_df["val/mAP_50_95"], label="mAP50-95")
ax.set_title("mAP")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
out = ARTIFACTS / "training_curves.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## Validate on val split

In [ ]:
from pathlib import Path
import numpy as np
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano

OUTPUT_DIR  = Path("../models/runs/rfdetr-nano-map50").resolve()
DATASET_DIR = Path("../data/labeled/race-vision.v1i.yolov8").resolve()

# Low threshold for mAP: let supervision sweep the full precision-recall curve.
# CONF_THRESHOLD=0.5 (used elsewhere for precision/recall at a fixed operating point)
# truncates the recall axis and badly underreports mAP.
MAP_THRESHOLD = 0.01

model_eval = RFDETRNano(pretrain_weights=str(OUTPUT_DIR / "checkpoint_best_ema.pth"))
print(f"Loaded: checkpoint_best_ema.pth")

val_img_dir = DATASET_DIR / "valid" / "images"
val_lbl_dir = DATASET_DIR / "valid" / "labels"
val_images  = sorted(val_img_dir.glob("*.*"))
print(f"Val images: {len(val_images)}")


def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return np.zeros((0, 4))
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1 = (cx - bw / 2) * img_w
            y1 = (cy - bh / 2) * img_h
            x2 = (cx + bw / 2) * img_w
            y2 = (cy + bh / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return np.array(boxes) if boxes else np.zeros((0, 4))


map_metric = sv.metrics.MeanAveragePrecision()

for img_path in val_images:
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size
    predictions = model_eval.predict(img, threshold=MAP_THRESHOLD)
    lbl_path = val_lbl_dir / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)
    targets = sv.Detections(
        xyxy=gt_boxes,
        class_id=np.zeros(len(gt_boxes), dtype=int) if len(gt_boxes) else np.zeros(0, dtype=int),
    )
    map_metric.update([predictions], [targets])

result = map_metric.compute()
print(f"\nmAP50:    {result.map50:.3f}")
print(f"mAP50-95: {result.map50_95:.3f}")

## Spot-check: inference on test images

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
from pathlib import Path
from PIL import Image
from rfdetr import RFDETRNano

OUTPUT_DIR  = Path("../models/runs/rfdetr-nano-map50").resolve()
DATASET_DIR = Path("../data/labeled/race-vision.v1i.yolov8").resolve()
ARTIFACTS   = Path("artifacts/rfdetr-nano-map50")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CONF_THRESHOLD = 0.5

if "model_eval" not in dir():
    model_eval = RFDETRNano(pretrain_weights=str(OUTPUT_DIR / "checkpoint_best_ema.pth"))

test_images = sorted((DATASET_DIR / "test" / "images").glob("*.*"))[:9]
print(f"Test images: {len(test_images)}")

annotator       = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

COLS = 3
rows = math.ceil(len(test_images) / COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 4))
axes = np.array(axes).flatten()

for ax, img_path in zip(axes, test_images):
    img = Image.open(img_path).convert("RGB")
    detections = model_eval.predict(img, threshold=CONF_THRESHOLD)

    labels = [f"bib {c:.2f}" for c in detections.confidence] if detections.confidence is not None else []
    annotated = annotator.annotate(scene=np.array(img).copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

    ax.imshow(annotated)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")

for ax in axes[len(test_images):]:
    ax.axis("off")

plt.tight_layout()
out = ARTIFACTS / "test_predictions.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## Summary

| Artifact | Path |
|---|---|
| Best weights (EMA) | `models/runs/rfdetr-nano-map50/checkpoint_best_ema.pth` |
| Training curves | `notebooks/artifacts/rfdetr-nano-map50/training_curves.png` |
| Test predictions | `notebooks/artifacts/rfdetr-nano-map50/test_predictions.png` |

**Checkpoint metric:** `val/mAP_50` (patched from default `val/mAP_50_95`).  
**Best checkpoint:** `checkpoint_best_ema.pth` — epoch 11, val/ema_mAP_50 = 0.970.  
**Patch location:** `.venv/lib/python3.12/site-packages/rfdetr/training/trainer.py`  
Next step: `06_rfdetr_evaluation.ipynb` — full test-split evaluation with the EMA checkpoint.